# Lab 3 — Millions of TVs, Who Is Watching?
Name: Yeremenko Anton | Group: KM-41

Setup

In [27]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab1-tv-panel/versions/4")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")

## Task 1.1 — Five questions
# **Answer:** the table (acr_sessions vs Lab 1 viewings) and the biggest difference.
# acr_sessions ; viewings
# Grain: a session ; a session
# Key: tv_id, ip ; respondent_id, resp_type
# Coverage: sees sessions, misses people ; sees both sessions and people
# Weight: weight = 1 ; weight = 1
# Provenance: synthetic ; synthetic
#
# The biggest difference is that "viewings" sees people watching and can identify them, while acr_sessions cant.

## Task 1.2 — Size and shape

In [124]:
acrcpy = acr_sessions
acrcpy["start_utc"] = acrcpy.start_utc - pd.Timedelta(hours=6)
acrcpy["end_utc"] = acrcpy.end_utc - pd.Timedelta(hours=6)
acrcpy["day"] = acrcpy.start_utc.dt.day
acrcpy["hrs"] = (acrcpy.end_utc - acrcpy.start_utc).dt.total_seconds() / 3600

rulehr = acrcpy[acrcpy.hrs >= 1/60].hrs.sum()
norulehr = acrcpy.hrs.sum()
intab = acrcpy[acrcpy.hrs >= 1/60].groupby("day").tv_id.nunique()
dropped = acrcpy[acrcpy.hrs < 1/60].session_id.nunique()
length = acrcpy["session_id"].nunique()

In [125]:
print("TVs:", acrcpy["tv_id"].nunique())
print("Sessions:", length)
print("Broadcast days:", 27-14+1)
print("Average TVs in-tab per day:", int(intab.mean()))
print("Hours per TV per day, raw:", round(norulehr/intab.sum(),2))
print("Hours per TV per day, after edit rule:", round(rulehr/intab.sum(),2))
print("Share of sessions dropped:", round(dropped/length,3))

TVs: 29940
Sessions: 6392981
Broadcast days: 14
Average TVs in-tab per day: 29174
Hours per TV per day, raw: 4.41
Hours per TV per day, after edit rule: 4.37
Share of sessions dropped: 0.299


## Task 1.3 — What the TV could not see

In [126]:
seshgroup = acrcpy.groupby("session_type").hrs.sum()
sourcegroup = acrcpy.groupby("input_source").hrs.sum()

In [127]:
# table: share of hours by session_type and by input_source
display(round(seshgroup/rulehr,3).to_frame())
display(round(sourcegroup/rulehr,3).to_frame())
print("Share of TV-on hours unknown (%):", round(seshgroup["unknown"]/rulehr*100,1))

,hrs
session_type,
app,0.257
dvr,0.025
live,0.327
unknown,0.398


,hrs
input_source,
antenna,0.087
built_in_app,0.308
cable_box,0.279
game_or_disc,0.024
satellite_box,0.077
streaming_device,0.231


Share of TV-on hours unknown (%): 39.8


**Answer:** why a real feed may have more unknown; two kinds of viewing that end
up unknown.
because some things wont be recognized, or will be mismatched due to a popup, or tricked by non-continuous feed like rewinds or pauses

## Task 2.1 — Match

In [128]:
merg = pd.merge(acrcpy,hh_attributes,on="hashed_ip")

tvs = ({"household_id": []}, {"amt": []})
tvs = merg.groupby("household_id").tv_id.nunique()

In [129]:
print("Matched to one household (%):", round(tvs[tvs == 1].sum() / tvs.sum() * 100 ,1))
print("Shared IP, unresolved (%):", round(tvs[tvs > 1].sum() / tvs.sum() * 100 ,1))
print("Not matched (%):", tvs[tvs == 0].count())

Matched to one household (%): 64.6
Shared IP, unresolved (%): 35.4
Not matched (%): 0


**Answer:** why an IP address is a weak key. - it doesnt distinguish between tvs

## Task 2.2 — Profile

In [130]:
... # table: group, footprint share, panel share, index (region and income)

Ellipsis

**Answer:** which groups are over/under-represented, and one reason.

## Task 2.3 — Ratings

In [131]:
... # table: telecast, TVs' HH rating, reference HH rating

Ellipsis

**Answer:** which are too high or low, and does it fit the profile?

## Task 3.1 — Hours per home

In [132]:
livebynet1 = acrcpy[acrcpy.session_type == "live"].groupby("network_code").hrs.sum()/14
viewings["hrs"] = (viewings.end_ts - viewings.start_ts).dt.total_seconds() / 3600
livebynet2 = viewings[viewings.hrs >= 1/60].groupby("network_code").hrs.sum()/28
merg2 = pd.merge(livebynet2,livebynet1,on="network_code",how="left")

In [133]:
print("Live hours per TV per day (ten networks):", round(merg2["hrs_y"],2))
print("Live hours per panel household per day (ten networks):", round(merg2["hrs_x"],2))

Live hours per TV per day (ten networks): network_code
ABC      4974.15
BRAVO     423.82
CBS      4632.82
FNC      2155.21
FOOD      620.85
FOX      5107.68
FX        718.93
HGTV      875.94
HIST      501.56
NBC      5422.04
Name: hrs_y, dtype: float64
Live hours per panel household per day (ten networks): network_code
ABC      3163.48
BRAVO     227.73
CBS      3456.71
FNC      1860.37
FOOD      390.75
FOX      3490.68
FX        370.98
HGTV      538.29
HIST      317.52
NBC      3581.11
Name: hrs_x, dtype: float64


**Answer:** which is lower and why. - panel data is lower by about 25%, presumably because its not as good at determining watchtime as smart tv screen reading algorithms.

## Task 3.2 — Network shares

In [134]:
merg2.columns = ["panel","smart"]
merg2["smart"] = merg2.smart / merg2.smart.sum()
merg2["panel"] = merg2.panel / merg2.panel.sum()
print(round(merg2,3)) # table: network, share in TVs, share in panel

              panel  smart
network_code              
ABC           0.182  0.196
BRAVO         0.013  0.017
CBS           0.199  0.182
FNC           0.107  0.085
FOOD          0.022  0.024
FOX           0.201  0.201
FX            0.021  0.028
HGTV          0.031  0.034
HIST          0.018  0.020
NBC           0.206  0.213


## Task 3.3 — What each can see

**Answer:** two or three things each way. - smart sees worse when the feed is altered somehow (popups,pauses,rewinds,etc), but it can see better what is it that is shown on the screen. meanwhile panel doesnt care about obstructions, but it cant see whats shown, and it has limitations to the type of session (live).

## Task 4.1 — Daily plots

In [157]:
big5 = ["NBC","FOX","ABC","CBS","FNC"]
live5 = acrcpy[acrcpy.session_type == "live"][acrcpy.network_code.isin(big5)]

daysums = pd.DataFrame()
for i in range(0,5):
    daysums[big5[i]] = live5[live5.network_code == big5[i]].groupby("day").hrs.sum()

/tmp/ipykernel_727/3194961897.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  live5 = acrcpy[acrcpy.session_type == "live"][acrcpy.network_code.isin(big5)]


In [159]:
print(round(daysums,2)) # plot: TVs in-tab per day; live hours per day for the big five networks

         NBC       FOX      ABC      CBS      FNC
day                                              
13   1851.80   2144.04  1363.05  1630.20   847.18
14   5015.05   3482.13  9151.74  4183.37  1992.45
15   5143.22   3476.64  4066.62  4689.31  2220.47
16   4401.74   3389.03  2828.93  4214.90  2316.05
17   4457.36   3637.40  3621.82  3905.22  2193.20
18   4940.42   9210.55  6403.94  5289.26  2179.36
19   6980.96   7959.94  5139.65  7353.83  2294.45
20   8305.26   3555.86  4355.83  4600.58  2311.70
21   4254.26   2558.53  8815.71  3005.99  1754.98
22   3554.44   2817.70  3083.50  3569.75  1687.22
23   4431.60   3462.90  2865.71  4346.15  2323.26
24   4450.64   3715.70  3527.23  3885.12  2201.36
25   5040.84  10609.10  6312.73  5183.03  2195.48
26   6790.10   9387.38  5147.73  6131.31  2250.95
27   6290.94   2100.59  2953.96  2871.44  1404.86


## Task 4.2 — Spike and broken day

In [160]:
print("Biggest spike (date, network):", "Fox on 25th")
print("Broken day (date):", ...)

Biggest spike (date, network): Fox on 25th
Broken day (date): Ellipsis


**Answer:** what aired; what went wrong with the data; drop, fix or flag? - 14th and 21st are mondays, and on ABC the watchtime is higher which correlates to football, same with 25th for fox - being the World Series

## Task 5.1 — Sampling error vs bias

In [209]:
merg3 = pd.merge(reference_ratings,programs[programs.program_name == "Hannity"],on="telecast_id",how="left")
ids = list(merg3["telecast_id"])
rates = list(merg3["hh_rating"])
ses = []
for i in range(0,len(ids)):
    cday = list(programs[programs.telecast_id == ids[i]].start_ts.dt.day)[0]
    amt1 = acrcpy[acrcpy.hrs >= 1/60]; amt1 = amt1[amt1.day == cday].tv_id.nunique()
    viewings["day"] = viewings.start_ts.dt.day
    amt2 = viewings[viewings.hrs >= 1/60]; amt2 = amt2[amt2.day == cday].respondent_id.nunique()
    se1 = (rates[i]/100 * (1 - rates[i]/100) / amt1)**.5
    se2 = (rates[i]/100 * (1 - rates[i]/100) / amt2)**.5
    ses.append([se1,se2])

In [212]:
print("smart panel ref")
for i in range(0,len(ids)):
    print(round(ses[i][0],3), round(ses[i][1],3), round(rates[i]/100,3)) 
    # table: telecast, SE footprint, SE panel, TVs' rating - reference

smart panel ref
0.003 0.005 0.259
0.003 0.005 0.331
0.002 0.004 0.106
0.001 0.002 0.023
0.001 0.002 0.037
0.001 0.001 0.016
0.001 0.002 0.035
0.001 0.001 0.012


**Answer:** which error is bigger, and does more data shrink it? - more data does indeed shrink the error, as with smart tvs the error is smaller than with panel

## Task 5.2 — Answer to the planner

**Answer:** 5-7 sentences. - it would be best if we had analysed data from earlier: from same days of the week and hour by hour in the days, to get the picture of how much more people actually would watch [whatever's being shown] with as little noise as possible. i would say that the standard error of a household rating is 0.002 on average, as several smaller ones still dont overshadow the bigger errors on some telecasts.

## Bonus

In [139]:
print("VPVH A25-54:", ...)
print("A25-54 rating (persons):", ...)

VPVH A25-54: Ellipsis
A25-54 rating (persons): Ellipsis


**Answer:** what this assumes.